<a href="https://colab.research.google.com/github/seu-usuario/seu-repo/blob/main/seaborn_saas_GABARITO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📈 Visualização Estatística de Dados com Seaborn — SaaS — GABARITO

**Contexto:** Analista de SaaS. Base com `idade, plano, tempo_uso_horas, bugs_reportados, satisfacao` (300 usuários) para mapear engajamento, uso por plano e correlação bugs × satisfação.

**Colab:** `Arquivo > Fazer upload do notebook`, executar com `Shift + Enter`. `seaborn`, `pandas`, `numpy`, `matplotlib` já vêm no Colab.

---

## 🧱 1. Configuração do Ambiente e Base de Dados

▶️ Execute primeiro — cria o DataFrame sintético com correlações visuais simuladas.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuração de estilo global do Seaborn
sns.set_theme(style="whitegrid", palette="muted")

# Geração de dados sintéticos
np.random.seed(101)
n = 300

dados = {
    'idade': np.random.normal(35, 10, n).astype(int),
    'plano': np.random.choice(['Gratuito', 'Básico', 'Pro'], n, p=[0.5, 0.3, 0.2]),
    'tempo_uso_horas': np.random.uniform(1, 50, n),
    'bugs_reportados': np.random.poisson(2, n),
    'satisfacao': np.random.randint(1, 11, n)
}

df_sistema = pd.DataFrame(dados)

# Ajustando regras de negócio sintéticas para gerar correlações visuais
df_sistema.loc[df_sistema['plano'] == 'Pro', 'tempo_uso_horas'] += 15
df_sistema.loc[df_sistema['plano'] == 'Pro', 'satisfacao'] += 2
df_sistema['satisfacao'] = df_sistema['satisfacao'] - (df_sistema['bugs_reportados'] * 0.5)
df_sistema['satisfacao'] = df_sistema['satisfacao'].clip(1, 10).astype(int)
df_sistema['idade'] = df_sistema['idade'].clip(18, 70)

print("Dataset gerado:", df_sistema.shape)
df_sistema.head()

print("\nContagem por plano:")
print(df_sistema['plano'].value_counts())

## 🟢 Parte 1 — Distribuições e Contagens (Básico)

### Tarefa 1: `countplot` de usuários por plano (ordenado por frequência)

In [ ]:
ordem_planos = df_sistema['plano'].value_counts().index  # decrescente por frequência

plt.figure(figsize=(8, 5))
sns.countplot(data=df_sistema, x='plano', order=ordem_planos)
plt.title('Contagem de Usuários por Plano (frequência decrescente)', fontsize=14, fontweight='bold')
plt.xlabel('Plano de Assinatura')
plt.ylabel('Nº de usuários')
plt.tight_layout()
plt.show()

### Tarefa 2: `histplot` da `idade` com KDE

In [ ]:
plt.figure(figsize=(9, 5))
sns.histplot(data=df_sistema, x='idade', bins=20, kde=True, color='steelblue')
plt.title('Distribuição da Idade dos Usuários', fontsize=14, fontweight='bold')
plt.xlabel('Idade (anos)')
plt.ylabel('Frequência')
plt.tight_layout()
plt.show()

print(f"Idade média: {df_sistema['idade'].mean():.1f} | mediana: {df_sistema['idade'].median():.0f}")

## 🟡 Parte 2 — Relações Categóricas e Numéricas (Intermediário)

### Tarefa 3: `boxplot` de `tempo_uso_horas` por `plano`

In [ ]:
# Apoio numérico
print(df_sistema.groupby('plano')['tempo_uso_horas'].describe().round(1))

plt.figure(figsize=(8, 5))
sns.boxplot(data=df_sistema, x='plano', y='tempo_uso_horas', order=ordem_planos)
plt.title('Tempo de Uso por Plano de Assinatura', fontsize=14, fontweight='bold')
plt.xlabel('Plano')
plt.ylabel('Tempo de uso (horas)')
plt.tight_layout()
plt.show()

# 💡 Resposta: Pro consome mais horas (regra sintética +15h); Gratuito o menor tempo.

### Tarefa 4: `violinplot` de `plano` × `satisfacao`

In [ ]:
plt.figure(figsize=(8, 5))
sns.violinplot(data=df_sistema, x='plano', y='satisfacao', order=ordem_planos)
plt.title('Densidade da Satisfação por Plano', fontsize=14, fontweight='bold')
plt.xlabel('Plano')
plt.ylabel('Satisfação (1–10)')
plt.tight_layout()
plt.show()

# 💡 Veja a largura da "viola": zonas mais largas = mais clientes com aquela nota.
# Pro tende a notas mais altas para o mesmo número de bugs (regra +2).

## 🔵 Parte 3 — Correlações e Análise Multivariada (Avançado)

### Tarefa 5: `scatterplot` tempo_uso_horas × satisfacao, `hue=plano`, `size=bugs_reportados`

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df_sistema,
                x='tempo_uso_horas', y='satisfacao',
                hue='plano', size='bugs_reportados',
                sizes=(20, 300), alpha=0.7)
plt.title('Engajamento × Satisfação (tamanho = bugs reportados)', fontsize=13, fontweight='bold')
plt.xlabel('Tempo de uso (horas)')
plt.ylabel('Satisfação (1–10)')
plt.tight_layout()
plt.show()

# 💡 Leitura: grandes volumes de bugs → pontos grandes e satisfação menor (Δ ≈ -0,5 por bug).

### Tarefa 6: Matriz de correlação de Pearson + `heatmap` divergente

In [ ]:
# Matriz de correlação (só variáveis numéricas)
corr = df_sistema.select_dtypes(include=[np.number]).corr(method='pearson')
print(corr.round(3))

# Heatmap divergente
plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, cmap='coolwarm', center=0, fmt='.2f',
            linewidths=0.5, vmin=-1, vmax=1)
plt.title('Matriz de Correlação de Pearson (variáveis numéricas)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# 💡 Esperados: satisfacao ~ negativo com bugs_reportados (-0,5);
# tempo_uso_horas ~ modesto com satisfacao (forte com plano, mas plano é categórico → fora da matriz).

### 🎁 Tarefa 7 (Bônus): `pairplot` com `hue=plano`

In [ ]:
g = sns.pairplot(data=df_sistema[['tempo_uso_horas', 'bugs_reportados', 'satisfacao', 'plano']],
                 hue='plano', kind='scatter', diag_kind='kde', palette='muted')
g.fig.suptitle('Pairplot: Engajamento, Bugs e Satisfação por Plano', y=1.02, fontweight='bold')
plt.show()

## 🎓 Conclusão executiva

In [ ]:
print("=" * 60)
print("RELATÓRIO SAAS — Perfil de Engajamento")
print("=" * 60)
print(f"Maior base: {df_sistema['plano'].value_counts().idxmax()} ({df_sistema['plano'].value_counts().max()} usuários)")
print(f"Plano mais engajado (horas): {df_sistema.groupby('plano')['tempo_uso_horas'].mean().idxmax()}")
print(f"Satisfação média: {df_sistema['satisfacao'].mean():.2f} | Mediana: {df_sistema['satisfacao'].median():.0f}")
print(f"Correlação bugs × satisfação: {corr.loc['bugs_reportados','satisfacao']:.2f}")
print("Recomendação: priorizar redução de bugs — cada bug derruba ~0,5 ponto de satisfação.")
print("=" * 60)

### 🚀 Desafio extra
1. `lmplot` com tendência por plano: `sns.lmplot(data=df_sistema, x='tempo_uso_horas', y='satisfacao', hue='plano')`
2. `catplot` com box por plano e `col='plano'`.
3. Salve o heatmap: `plt.savefig('corr.png', dpi=300, bbox_inches='tight')`.